**This vizualization is based on the following study and data analysis problem:**

Ferreira, N., Fisher, D., & Konig, A. C. (2014, April). [Sample-oriented task-driven visualizations: allowing users to make better, more confident decisions.](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/Ferreira_Fisher_Sample_Oriented_Tasks.pdf)       

In Proceedings of the SIGCHI Conference on Human Factors in Computing Systems (pp. 571-580). ACM. ([video](https://www.youtube.com/watch?v=BI7GAs-va-Q&themeRefresh=1))

In this paper the authors describe the challenges users face when trying to make judgements about probabilistic data generated through samples. As an example, they look at a bar chart of four years of data. Each year has a y-axis value, which is derived from a sample of a larger dataset. For instance, the first value might be the number votes in a given district or riding for 1992, with the average being around 33,000. On top of this is plotted the 95% confidence interval for the mean.

A challenge that users face is that, for a given y-axis value (e.g. 42,000), it is difficult to know which x-axis values are most likely to be representative, because the confidence levels overlap and their distributions are different (the lengths of the confidence interval bars are unequal). One of the solutions the authors propose for this problem is to allow users to indicate the y-axis value of interest (e.g. 42,000) and then draw a horizontal line and color bars based on this value. So bars might be colored red if they are definitely above this value (given the confidence interval), blue if they are definitely below this value, or white if they contain this value.

In [1]:
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors

import pandas as pd
import numpy as np
from scipy import stats
import math

In [2]:
# Generate sample data
np.random.seed(12345)

df = pd.DataFrame([np.random.normal(32000,200000,3650), 
                   np.random.normal(43000,100000,3650), 
                   np.random.normal(43500,140000,3650), 
                   np.random.normal(48000,70000,3650)], 
                  index=[1992,1993,1994,1995])
df

,0,1,2,3,4,5,6,7,8,9,...,3640,3641,3642,3643,3644,3645,3646,3647,3648,3649
1992,-8941.531897,127788.667612,-71887.743011,-79146.060869,425156.114501,310681.166595,50581.575349,88349.230566,185804.513522,281286.947277,...,171938.760289,150650.759924,203663.976475,-377877.158072,-197214.093861,24185.008589,-56826.729535,-67319.766489,113377.299342,-4494.878538
1993,-51896.094813,198350.518755,-123518.252821,-129916.759685,216119.147314,49845.883728,149135.648505,62807.672113,23365.577348,-109686.264981,...,-44566.520071,101032.122475,117648.199945,160475.622607,-13759.888342,-37333.493572,103019.841174,179746.127403,13455.493990,34442.898855
1994,152336.932066,192947.128056,389950.263156,-93006.152024,100818.575896,5529.230706,-32989.370488,223942.967178,-66721.580898,47826.269111,...,165085.806360,74735.174090,107329.726875,199250.734156,-36792.202754,-71861.846997,26375.113219,-29328.078384,65858.761714,-91542.001049
1995,-69708.439062,-13289.977022,-30178.390991,55052.181256,152883.621657,12930.835194,63700.461932,64148.489835,-29316.268556,59645.677367,...,-13901.388118,50173.686673,53965.990717,4128.990173,72202.595138,39937.199964,139472.114293,59386.186379,73362.229590,28705.082908


In [3]:
# Create second datafram containing statistics computations of the generated data
df2 = pd.DataFrame(index=df.index)

df2['Size'] = df.apply(lambda x: len(x), axis=1)
df2['Mean'] = df.apply(np.mean, axis=1)
df2['STD'] = df.apply(np.std, axis=1)
df2['STE'] = df.apply(stats.sem, axis=1)

confidence = 0.95
    
c_intervals = df2.apply(lambda x: stats.t.interval(confidence, df=x['Size']-1, loc=x['Mean'], scale=x['STE']), axis=1)
df2['CI_lower'], df2['CI_upper'] = zip(*c_intervals)
df2

,Size,Mean,STD,STE,CI_lower,CI_upper
1992,3650,33312.107476,200603.415985,3320.866311,26801.169458,39823.045494
1993,3650,41861.859541,98384.876053,1628.701180,38668.604697,45055.114385
1994,3650,39493.304941,140350.695166,2323.419534,34937.975350,44048.634533
1995,3650,47743.550969,69771.625748,1155.026400,45478.989678,50008.112260


In [4]:
# Specify plotting data
yVals = np.array(df2['Mean'])
# print(yVals)
xVals = df2.index.tolist()
# print(xVals)
error = np.array([df2['Mean']-df2['CI_lower'], df2['CI_upper']-df2['Mean']], dtype='float64')

In [5]:
# Create Custom Colormap
custom_cmap = colors.LinearSegmentedColormap.from_list("custom", ['royalblue','white', 'firebrick'])
color_pic = cm.ScalarMappable(colors.Normalize(0, 1), custom_cmap)
color_pic.set_array([])

**Note: Unfortunately, the full interactive capabilities cannot function in the Kaggle Public Display View. If you would like to have a better look at how `interact()` works in this section of the notebook, click on the *Copy and Edit* button above to enter the Kernel View. Another option is to download this notebook and run it on your local environment.**

In [6]:
# Create slider widgets
@interact(
    # Specifying the handler for the n argument in the function
    n = widgets.IntSlider(min=2, max=50000, step=1, value=39500)
)

# Graph plotting function
def plot(n):
    y_val = n
    fig, ax = plt.subplots(figsize=(10,6))
    
    # Colorbar
    color_bar = fig.colorbar(color_pic, ticks=[0,0.5,1], ax=ax, location='right', orientation='vertical')
    color_bar.set_ticklabels(['Mean likely below y',
                     'Mean likely at y',
                     'Mean likely above y'])

    # Assign Bar Color based on slider value
    color_condition = []
    color_condition.extend(df2.apply(lambda x: 1 - stats.norm.cdf(y_val, loc=x['Mean'], scale=x['STE']), axis=1).tolist())

    # Plot Graph
    plt.bar(xVals, yVals, yerr=error, 
            capsize=10, color=color_pic.to_rgba(color_condition), 
            edgecolor='k', align='center')

    plt.axhline(y=y_val, c='k', lw=1, label="y="+str(round(y_val)))
    plt.xticks(xVals)

    # Hide the right and top spines
    for position in ['top', 'right', 'bottom', 'left']:
        ax.spines[position].set_visible(False)
        
    # Label horizontal line
    plt.gca().annotate("y="+str(round(y_val)), xy=(1995.45, y_val+600), fontsize=8, weight="bold")

interactive(children=(IntSlider(value=39500, description='n', max=50000, min=2), Output()), _dom_classes=('wid…

**Note: Unfortunately, the full interactive capabilities cannot function in the Kaggle Public Display View. If you would like to have a better look at how `interact()` works in this section of the notebook, click on the *Copy and Edit* button above to enter the Kernel View. Another option is to download this notebook and run it on your local environment.**